In [5]:
import sys

In [4]:
from typing import List, Dict, Any
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from neo4j import GraphDatabase

In [7]:
# Neo4j 연결 설정
NEO4J_URI = "bolt://neo4j-gds-apoc-n10s:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "neo4jpassword"

# Neo4j 연결 테스트
try:
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as test_driver:
        with test_driver.session() as session:
            result = session.run("MATCH (n) RETURN count(n) as count")
            count = result.single()["count"]
            print(f"Neo4j 연결 성공! 데이터베이스에 있는 노드 수: {count}")
except Exception as e:
    print(f"Neo4j 연결 실패: {str(e)}")

Neo4j 연결 성공! 데이터베이스에 있는 노드 수: 332


In [23]:
from langchain_community.graphs import Neo4jGraph

# Neo4jGraph 객체 생성
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    enhanced_schema=True,
)

schema = graph.get_schema
print("Neo4j Schema:")
print(schema)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Neo4j Schema:
Node properties:
- **요금제**
  - `장애인혜택제공여부`: STRING Available options: ['N', 'Y']
  - `고객유형별가입가능여부`: STRING Available options: ['null', 'true', 'false']
  - `장애인부가통화추가제공량`: INTEGER Min: 0, Max: 250
  - `문자대상`: STRING Available options: ['[]']
  - `가입가능최소나이계산기준`: STRING Available options: ['null', '일기준', '월기준']
  - `다이렉트플랜가입가능여부`: STRING Available options: ['Y', 'null']
  - `가입가능최대나이`: INTEGER Min: 12, Max: 999
  - `개인고객세부유형별가입가능여부`: STRING Available options: ['null', 'true']
  - `최대나이계산기준`: STRING Available options: ['null', '월기준']
  - `T지원금약정동시가입가능여부`: STRING Available options: ['Y', 'null']
  - `선택약정동시가입가능여부`: STRING Available options: ['Y', 'null']
  - `군인전용요금제여부`: STRING Available options: ['null', 'Y']
  - `가입가능최소나이`: INTEGER Min: 0, Max: 80
  - `동일명의가입가능여부`: STRING Available options: ['null', 'false']
  - `최소충전금액`: FLOAT Min: 0.0, Max: 1000.0
  - `최대충전금액`: FLOAT Min: 0.0, Max: 20000.0
  - `시니어대상데이터소진후최대금액및속도제한적용`: STRING Available options: ['N', 'Y']
  - `기본제공데이터용량`:

In [10]:
# LLM 모델 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key="e97ee307-a791-4e06-ade1-df4b9d032eed",
    openai_api_base="https://aihub-api.sktelecom.com/aihub/v2/sandbox",
    streaming=True,
    temperature=0
)

# Neo4j 드라이버 초기화
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [11]:
# 상태 타입 정의
class State:
    def __init__(self):
        self.messages: List = []
        self.steps: List[str] = []
        self.current_step: int = 0
        self.results: Dict = {}
        self.final_answer: Dict = {}

# 질문 분석 및 스텝 생성 함수
def analyze_question(state: State) -> State:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "사용자의 모바일 요금제 관련 질문을 분석하여 필요한 검색 단계들을 생성하세요."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    response = llm.invoke(prompt.format_messages(messages=state.messages))
    steps = response.content.split('\n')
    
    state.steps = steps
    return state

In [13]:
# 테스트를 위한 상태 객체 생성
state = State()
state.messages = [
    HumanMessage(content="가장 인기 많은 무제한 요금제 알려줘")
]

# analyze_question 함수 테스트
try:
    result_state = analyze_question(state)
    print("분석된 검색 단계:")
    for i, step in enumerate(result_state.steps):
        print(f"{i+1}. {step}")
except Exception as e:
    print(f"에러 발생: {str(e)}")


분석된 검색 단계:
1. 무제한 요금제에 대한 정보를 찾기 위해 다음과 같은 검색 단계를 진행할 수 있습니다:
2. 
3. 1. **무제한 요금제의 정의 확인**: 무제한 요금제가 무엇인지, 어떤 서비스가 포함되는지 이해하기.
4.    
5. 2. **국가 및 지역 선택**: 무제한 요금제를 찾고자 하는 국가나 지역을 결정하기. (예: 한국, 미국 등)
6. 
7. 3. **이통사 비교**: 해당 지역에서 제공하는 주요 이동통신사(예: SKT, KT, LG U+ 등)의 무제한 요금제 목록 확인하기.
8. 
9. 4. **요금제 특징 분석**: 각 요금제의 가격, 데이터 속도, 추가 혜택(예: 통화, 문자, 해외 로밍 등) 비교하기.
10. 
11. 5. **사용자 리뷰 및 평점 확인**: 인기 있는 무제한 요금제에 대한 사용자 리뷰 및 평점 검색하기.
12. 
13. 6. **프로모션 및 할인 정보 확인**: 현재 진행 중인 프로모션이나 할인 혜택이 있는지 확인하기.
14. 
15. 7. **결정 및 선택**: 수집한 정보를 바탕으로 가장 인기 있는 무제한 요금제를 결정하기.
16. 
17. 이러한 단계를 통해 원하는 정보를 효과적으로 찾을 수 있습니다.


In [ ]:

# Cypher 쿼리 생성 및 실행 함수
def execute_step(state: State) -> State:
    current_step = state.steps[state.current_step]
    
    # Cypher 쿼리 생성
    prompt = ChatPromptTemplate.from_messages([
        ("system", "주어진 검색 단계에 대한 Neo4j Cypher 쿼리를 생성하세요."),
        ("user", f"단계: {current_step}")
    ])
    
    query_response = llm.invoke(prompt.format_messages())
    cypher_query = query_response.content
    
    # Neo4j 쿼리 실행
    with driver.session() as session:
        result = session.run(cypher_query).data()
        state.results[state.current_step] = result
    
    return state

# 결과 검증 함수
def validate_result(state: State) -> Dict[str, Any]:
    current_step = state.steps[state.current_step]
    current_result = state.results[state.current_step]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "현재 단계의 검색 결과가 적절한지 검증하세요."),
        ("user", f"단계: {current_step}\n결과: {current_result}")
    ])
    
    validation = llm.invoke(prompt.format_messages())
    is_valid = "유효함" in validation.content
    
    if is_valid:
        if state.current_step < len(state.steps) - 1:
            state.current_step += 1
            return {"next": "execute_step"}
        else:
            return {"next": "final_answer"}
    else:
        return {"next": "execute_step"}

# 최종 답변 생성 함수
def generate_final_answer(state: State) -> State:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "모든 검색 결과를 종합하여 최종 답변을 JSON 형식으로 생성하세요."),
        ("user", f"검색 결과: {state.results}")
    ])
    
    response = llm.invoke(prompt.format_messages())
    state.final_answer = eval(response.content)
    return state

# 워크플로우 그래프 생성
workflow = StateGraph(State)

# 노드 추가
workflow.add_node("analyze_question", analyze_question)
workflow.add_node("execute_step", execute_step)
workflow.add_node("validate_result", validate_result)
workflow.add_node("final_answer", generate_final_answer)

# 엣지 연결
workflow.set_entry_point("analyze_question")
workflow.add_edge("analyze_question", "execute_step")
workflow.add_edge("execute_step", "validate_result")
workflow.add_conditional_edges(
    "validate_result",
    {
        "execute_step": lambda x: x["next"] == "execute_step",
        "final_answer": lambda x: x["next"] == "final_answer"
    }
)
workflow.add_edge("final_answer", END)

# 그래프 컴파일
app = workflow.compile()

# 사용 예시
def search_mobile_plans(question: str) -> Dict:
    state = State()
    state.messages = [HumanMessage(content=question)]
    result = app.invoke(state)
    return result.final_answer
